# 각종 설정
모델 하이퍼파라메터(hyperparameter)와 저장 위치 등 설정 정보를 선언합니다.

In [6]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.device_count())

2.14.1+cpu
False
0


In [4]:
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)
from datasets import load_dataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"학습 장치: {device}")

if device.type == "cuda":
    print(f"학습 GPU: {torch.cuda.get_device_name(0)}")

# 1. 모델명 및 경로 설정
MODEL_NAME = "beomi/kcbert-base"
OUTPUT_DIR = "./checkpoint-doccls"
MAX_SEQ_LENGTH = 128

# 2. 토크나이저 및 모델 로드
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

# 3. NSMC 데이터셋 로드 (허깅페이스 데이터셋 스크립트 차단 우회를 위해 원본 TSV 주소에서 직접 로드)
dataset = load_dataset(
    "csv",
    data_files={
        "train": "https://raw.githubusercontent.com/e9t/nsmc/master/ratings_train.txt",
        "test": "https://raw.githubusercontent.com/e9t/nsmc/master/ratings_test.txt"
    },
    delimiter="\t"
)

# 결측치(Null)가 있는 행 제거 및 document 컬럼을 string 타입으로 변환
dataset = dataset.filter(lambda x: x['document'] is not None)

# 4. 데이터 전처리(토크나이제이션) 함수 정의
def tokenize_function(examples):
    return tokenizer(
        examples["document"],
        max_length=MAX_SEQ_LENGTH,
        padding="max_length",
        truncation=True
    )

# 데이터셋에 토큰화 적용
tokenized_datasets = dataset.map(tokenize_function, batched=True)

# 5. TrainingArguments 설정
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    learning_rate=5e-5,
    num_train_epochs=3,
    seed=7,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    logging_steps=500,
)

# 6. Trainer 초기화 및 학습 실행 (최신 버전에 맞추어 tokenizer 대신 processing_class를 사용합니다)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    processing_class=tokenizer,
)

# 학습 시작
trainer.train()
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

학습 장치: cpu


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 934.71it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: beomi/kcbert-base
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoin

KeyboardInterrupt: 

In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_DIR = "./checkpoint-doccls"
MAX_SEQ_LENGTH = 128

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR)
model.to(device)
model.eval()

def predict_sentiment(text):
    inputs = tokenizer(
        text,
        max_length=MAX_SEQ_LENGTH,
        truncation=True,
        padding=True,
        return_tensors="pt",
    )
    inputs = {key: value.to(device) for key, value in inputs.items()}

    with torch.inference_mode():
        logits = model(**inputs).logits
        probabilities = torch.softmax(logits, dim=-1)[0]

    predicted_id = probabilities.argmax().item()
    labels = {0: "부정", 1: "긍정"}

    return {
        "sentiment": labels[predicted_id],
        "confidence": probabilities[predicted_id].item(),
        "negative_probability": probabilities[0].item(),
        "positive_probability": probabilities[1].item(),
    }

result = predict_sentiment("기대 안 했는데 정말 재미있었어요!")
print(result)

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 785.87it/s]


{'sentiment': '긍정', 'confidence': 0.9930096864700317, 'negative_probability': 0.006990361027419567, 'positive_probability': 0.9930096864700317}


In [3]:
import os

# 설정한 출력 경로 확인
if os.path.exists(OUTPUT_DIR):
    print(f"[정상] '{OUTPUT_DIR}' 경로가 존재합니다.")
    print("\n--- 생성된 산출물 파일 목록 ---")
    for root, dirs, files in os.walk(OUTPUT_DIR):
        level = root.replace(OUTPUT_DIR, '').count(os.sep)
        indent = ' ' * 4 * (level)
        print(f"{indent}{os.path.basename(root)}/")
        subindent = ' ' * 4 * (level + 1)
        for f in files[:10]:  # 최대 10개만 출력
            print(f"{subindent}{f}")
else:
    print(f"[주의] '{OUTPUT_DIR}' 경로를 찾을 수 없습니다. 학습이 아직 완료되지 않았거나 경로 설정을 확인해 주세요.")

[정상] './checkpoint-doccls' 경로가 존재합니다.

--- 생성된 산출물 파일 목록 ---
checkpoint-doccls/
    config.json
    model.safetensors
    tokenizer.json
    tokenizer_config.json
    training_args.bin
    checkpoint-14064/
        config.json
        model.safetensors
        optimizer.pt
        rng_state.pth
        scheduler.pt
        tokenizer.json
        tokenizer_config.json
        trainer_state.json
        training_args.bin
    checkpoint-4688/
        config.json
        model.safetensors
        optimizer.pt
        rng_state.pth
        scheduler.pt
        tokenizer.json
        tokenizer_config.json
        trainer_state.json
        training_args.bin
    checkpoint-9376/
        config.json
        model.safetensors
        optimizer.pt
        rng_state.pth
        scheduler.pt
        tokenizer.json
        tokenizer_config.json
        trainer_state.json
        training_args.bin
